# Combine microns1412 and visp_deltalakes into a single Delta Lake dataset

Auto-discovers all delta tables in both source datasets by scanning for `_delta_log/` directories,
then writes a unified combined dataset to `/scratch/combined_datasets/`.

- Tables that exist at the same relative path in both sources are concatenated.
- Tables unique to one source are copied through as-is.
- Nested sub-tables (e.g. `cellfeatures/csm_cluster_features`) remain independent.

In [1]:
import os

import pyarrow as pa
import polars as pl
from deltalake import DeltaTable, write_deltalake

In [2]:
CODEOCEAN_MICRONS = "/data/microns1412"
CODEOCEAN_VISP = "/data/visp_deltalakes"
LOCAL_MICRONS = "../data/microns1412"
LOCAL_VISP = "../data/visp_deltalakes"

MICRONS_ROOT = CODEOCEAN_MICRONS if os.path.exists(CODEOCEAN_MICRONS) else LOCAL_MICRONS
VISP_ROOT = CODEOCEAN_VISP if os.path.exists(CODEOCEAN_VISP) else LOCAL_VISP

OUTPUT_ROOT = "/scratch/combined_datasets"

print(f"Microns root: {MICRONS_ROOT}")
print(f"VISP root:    {VISP_ROOT}")
print(f"Output root:  {OUTPUT_ROOT}")

Microns root: /data/microns1412
VISP root:    /data/visp_deltalakes
Output root:  /scratch/combined_datasets


In [ ]:
def discover_delta_tables(root: str) -> dict[str, str]:
    """
    Walk *root* and return {relative_table_path: absolute_table_path}
    for every directory that contains a `_delta_log/` subdirectory.
    """
    root = os.path.abspath(root)
    tables = {}
    for dirpath, dirnames, _ in os.walk(root):
        if "_delta_log" in dirnames:
            rel = os.path.relpath(dirpath, root)
            tables[rel] = dirpath
            dirnames.remove("_delta_log")
    return tables

In [4]:
microns_tables = discover_delta_tables(MICRONS_ROOT)
visp_tables = discover_delta_tables(VISP_ROOT)

all_rel_paths = sorted(set(microns_tables) | set(visp_tables))

table_sources: dict[str, list[str]] = {}
for rel in all_rel_paths:
    sources = []
    if rel in microns_tables:
        sources.append(microns_tables[rel])
    if rel in visp_tables:
        sources.append(visp_tables[rel])
    table_sources[rel] = sources

print(f"Discovered {len(all_rel_paths)} delta tables:\n")
for rel, srcs in table_sources.items():
    labels = []
    if rel in microns_tables:
        labels.append("microns")
    if rel in visp_tables:
        labels.append("visp")
    print(f"  {rel:50s}  [{', '.join(labels)}]")

Discovered 20 delta tables:

  .Trash-0/files/minnie65_std_transform_coordinates   [microns]
  cellcellconnectivitylong                            [microns]
  cellfeaturedefinition                               [microns]
  cellfeaturedefinition/visp_exc                      [visp]
  cellfeaturedefinition/visp_inh                      [visp]
  cellfeatures/csm_cluster_features                   [microns]
  cellfeatures/csm_cluster_features_umap              [microns]
  cellfeatures/exc_morph_features                     [visp]
  cellfeatures/inh_morph_features                     [visp]
  cellfeatures/minnie65_std_transform_coordinates     [microns]
  cellfeatureset                                      [microns]
  cellfeatureset/visp_exc                             [visp]
  cellfeatureset/visp_inh                             [visp]
  celltoclustermapping                                [visp]
  cluster                                             [microns, visp]
  clustermembership       

In [5]:
def combine_and_write(rel_path: str, source_paths: list[str], output_root: str) -> int:
    """
    Read delta table(s) from *source_paths*, concatenate if more than one,
    and write a fresh delta table to output_root/rel_path.

    Returns total row count of the combined table.
    """
    tables = []
    partition_cols: list[str] = []

    for src in source_paths:
        dt = DeltaTable(src)
        part_cols = dt.metadata().partition_columns
        if part_cols:
            partition_cols = part_cols
        tables.append(dt.to_pyarrow_table())

    if len(tables) == 1:
        combined = tables[0]
    else:
        combined = pa.concat_tables(tables, promote_options="permissive")

    out_path = os.path.join(output_root, rel_path)
    os.makedirs(out_path, exist_ok=True)

    write_kwargs = dict(mode="overwrite")
    if partition_cols:
        write_kwargs["partition_by"] = partition_cols

    write_deltalake(out_path, combined, **write_kwargs)
    return combined.num_rows

In [6]:
os.makedirs(OUTPUT_ROOT, exist_ok=True)

for rel_path, source_paths in table_sources.items():
    n_sources = len(source_paths)
    action = "merging" if n_sources > 1 else "copying"
    print(f"{action:8s} {rel_path} ({n_sources} source{'s' if n_sources > 1 else ''}) ... ", end="", flush=True)
    nrows = combine_and_write(rel_path, source_paths, OUTPUT_ROOT)
    print(f"{nrows:,} rows")

print("\nDone.")

copying  .Trash-0/files/minnie65_std_transform_coordinates (1 source) ... 133,969 rows
copying  cellcellconnectivitylong (1 source) ... 1,592,196 rows
copying  cellfeaturedefinition (1 source) ... 87 rows
copying  cellfeaturedefinition/visp_exc (1 source) ... 50 rows
copying  cellfeaturedefinition/visp_inh (1 source) ... 46 rows
copying  cellfeatures/csm_cluster_features (1 source) ... 35,787 rows
copying  cellfeatures/csm_cluster_features_umap (1 source) ... 35,787 rows
copying  cellfeatures/exc_morph_features (1 source) ... 734 rows
copying  cellfeatures/inh_morph_features (1 source) ... 520 rows
copying  cellfeatures/minnie65_std_transform_coordinates (1 source) ... 133,969 rows
copying  cellfeatureset (1 source) ... 3 rows
copying  cellfeatureset/visp_exc (1 source) ... 1 rows
copying  cellfeatureset/visp_inh (1 source) ... 1 rows
copying  celltoclustermapping (1 source) ... 18,171 rows
merging  cluster (2 sources) ... 202 rows
merging  clustermembership (2 sources) ... 109,977 row

## Verification

Read back the combined tables and print shapes to confirm everything was written correctly.

In [7]:
print(f"{'Table':<50s} {'Rows':>10s} {'Cols':>6s}")
print("-" * 68)

for rel_path in sorted(table_sources):
    out_path = os.path.join(OUTPUT_ROOT, rel_path)
    df = pl.read_delta(out_path)
    print(f"{rel_path:<50s} {df.shape[0]:>10,d} {df.shape[1]:>6d}")

Table                                                    Rows   Cols
--------------------------------------------------------------------
.Trash-0/files/minnie65_std_transform_coordinates     133,969      6
cellcellconnectivitylong                            1,592,196      9
cellfeaturedefinition                                      87      6
cellfeaturedefinition/visp_exc                             50      6
cellfeaturedefinition/visp_inh                             46      6
cellfeatures/csm_cluster_features                      35,787     85
cellfeatures/csm_cluster_features_umap                 35,787      5
cellfeatures/exc_morph_features                           734     53
cellfeatures/inh_morph_features                           520     49
cellfeatures/minnie65_std_transform_coordinates       133,969      6
cellfeatureset                                              3      4
cellfeatureset/visp_exc                                     1      4
cellfeatureset/visp_inh           

In [3]:
print("Sample: combined dataset table")
pl.read_delta(os.path.join(OUTPUT_ROOT, "dataset"))

Sample: combined dataset table


id,name,publication,modality,project_id
str,str,str,str,str
"""visp_exc_wnm""","""VISp excitatory whole neuron m…","""doi.org/10.1101/2023.11.25.568…","""MORPHOLOGY""","""visp_wnm"""
"""minnie65_v1412_csm_cluster""","""Minnie65 v1412 CSM Dendrite Ul…","""none""","""ELECTRON_MICROSCOPY""","""minnie65"""
"""minnie65_v1412_proofread""","""Minnie65 v1412 Cells With Proo…","""none""","""ELECTRON_MICROSCOPY""","""minnie65"""
"""visp_exc_patchseq""","""VISp excitatory Patch-seq data…","""doi.org/10.1101/2023.11.25.568…","""MORPHOLOGY""","""visp_patchseq"""
"""visp_inh_patchseq""","""VISp inhibitory Patch-seq data…","""doi.org/10.1016/j.cell.2020.09…","""MORPHOLOGY""","""visp_patchseq"""
"""tasic_2018_visp_scrnaseq""","""Tasic et al. 2018 VISp RNA-seq…","""doi.org/10.1038/s41586-018-065…","""OTHER""","""tasic_visp_scrnaseq"""


In [12]:
df = pl.read_delta(os.path.join(OUTPUT_ROOT, "dataitem_dataset_association"))
print(f"Unique project_ids: {df['project_id'].unique().to_list()}")
print(f"Unique dataset_ids: {df['dataset_id'].unique().to_list()}")
df

Unique project_ids: ['visp_wnm', 'minnie65', 'visp_patchseq']
Unique dataset_ids: ['minnie65_v1412_csm_cluster', 'visp_inh_patchseq', 'visp_exc_patchseq', 'minnie65_v1412_proofread', 'visp_exc_wnm']


dataitem_id,dataset_id,project_id
str,str,str
"""182709_6984-X2452-Y12423_reg""","""visp_exc_wnm""","""visp_wnm"""
"""182709_7126-X2913-Y10535_reg""","""visp_exc_wnm""","""visp_wnm"""
"""182724_5937-X3804-Y11955_reg""","""visp_exc_wnm""","""visp_wnm"""
"""182724_6175-X3782-Y10859_reg""","""visp_exc_wnm""","""visp_wnm"""
"""182724_6354-X4834-Y8105_reg""","""visp_exc_wnm""","""visp_wnm"""
…,…,…
"""700357513""","""visp_inh_patchseq""","""visp_patchseq"""
"""711511692""","""visp_inh_patchseq""","""visp_patchseq"""
"""652809932""","""visp_inh_patchseq""","""visp_patchseq"""


In [6]:
print("Sample: combined cluster table")
df = pl.read_delta(os.path.join(OUTPUT_ROOT, "cluster"))
print(f"Unique project_ids: {df['project_id'].unique().to_list()}")
df

Sample: combined cluster table
Unique project_ids: ['visp_met_types', 'tasic_2018_visp_scrnaseq', 'minnie65']


id,parent,children,level,score,hex_color,heirachy_category,distance_to_parent,project_id
str,str,list[str],i64,f64,str,str,f64,str
"""cell""",null,"[""GABAergic"", ""Glutamatergic"", ""Non-Neuronal""]",0,null,"""#000000""","""major_class""",null,"""tasic_2018_visp_scrnaseq"""
"""GABAergic""","""cell""","[""Pvalb"", ""Vip"", … ""Meis2""]",1,null,"""#EF4136""","""class""",null,"""tasic_2018_visp_scrnaseq"""
"""Glutamatergic""","""cell""","[""L4"", ""L2/3 IT"", … ""CR""]",1,null,"""#27AAE1""","""class""",null,"""tasic_2018_visp_scrnaseq"""
"""Non-Neuronal""","""cell""","[""Oligo"", ""Astro"", … ""SMC""]",1,null,"""#8D1800""","""class""",null,"""tasic_2018_visp_scrnaseq"""
"""Pvalb""","""GABAergic""","[""Pvalb Tpbg"", ""Pvalb Reln Itm2a"", … ""Pvalb Sema3e Kank4""]",2,null,"""#D93137""","""subclass""",null,"""tasic_2018_visp_scrnaseq"""
…,…,…,…,…,…,…,…,…
"""L5IT""","""glutamatergic""",null,2,null,"""#b76969""","""subtype""",null,"""minnie65"""
"""L6IT""","""glutamatergic""",null,2,null,"""#7d354c""","""subtype""",null,"""minnie65"""
"""L5ET""","""glutamatergic""",null,2,null,"""#c68076""","""subtype""",null,"""minnie65"""


In [5]:
print("Sample: combined dataitem table")
df = pl.read_delta(os.path.join(OUTPUT_ROOT, "dataitem"))
print(f"Unique project_ids: {df['project_id'].unique().to_list()}")
df

Sample: combined dataitem table
Unique project_ids: ['minnie65', 'visp_wnm', 'visp_patchseq']


id,name,neuroglancer_link,project_id
str,str,str,str
"""373879""","""864691136090135607""",null,"""minnie65"""
"""201858""","""864691135373893678""",null,"""minnie65"""
"""600774""","""864691135682378744""",null,"""minnie65"""
"""408486""","""864691135194387242""",null,"""minnie65"""
"""598774""","""864691135741608653""",null,"""minnie65"""
…,…,…,…
"""700357513""","""700357513""",null,"""visp_patchseq"""
"""711511692""","""711511692""",null,"""visp_patchseq"""
"""652809932""","""652809932""",null,"""visp_patchseq"""


---

# Clusters that were mapped to
__tasic_2018_visp_scrnaseq, visp_met_types, minnie65__

In [14]:
df = pl.read_delta(os.path.join(OUTPUT_ROOT, "cluster"))
print(f"Unique project_ids: {df['project_id'].unique().to_list()}")
df

Unique project_ids: ['minnie65', 'tasic_2018_visp_scrnaseq', 'visp_met_types']


id,parent,children,level,score,hex_color,heirachy_category,distance_to_parent,project_id
str,str,list[str],i64,f64,str,str,f64,str
"""cell""",null,"[""GABAergic"", ""Glutamatergic"", ""Non-Neuronal""]",0,null,"""#000000""","""major_class""",null,"""tasic_2018_visp_scrnaseq"""
"""GABAergic""","""cell""","[""Pvalb"", ""Vip"", … ""Meis2""]",1,null,"""#EF4136""","""class""",null,"""tasic_2018_visp_scrnaseq"""
"""Glutamatergic""","""cell""","[""L4"", ""L2/3 IT"", … ""CR""]",1,null,"""#27AAE1""","""class""",null,"""tasic_2018_visp_scrnaseq"""
"""Non-Neuronal""","""cell""","[""Oligo"", ""Astro"", … ""SMC""]",1,null,"""#8D1800""","""class""",null,"""tasic_2018_visp_scrnaseq"""
"""Pvalb""","""GABAergic""","[""Pvalb Tpbg"", ""Pvalb Reln Itm2a"", … ""Pvalb Sema3e Kank4""]",2,null,"""#D93137""","""subclass""",null,"""tasic_2018_visp_scrnaseq"""
…,…,…,…,…,…,…,…,…
"""L5IT""","""glutamatergic""",null,2,null,"""#b76969""","""subtype""",null,"""minnie65"""
"""L6IT""","""glutamatergic""",null,2,null,"""#7d354c""","""subtype""",null,"""minnie65"""
"""L5ET""","""glutamatergic""",null,2,null,"""#c68076""","""subtype""",null,"""minnie65"""


In [15]:
df_ps = df.filter(pl.col("project_id") == "visp_met_types")
df_ps

id,parent,children,level,score,hex_color,heirachy_category,distance_to_parent,project_id
str,str,list[str],i64,f64,str,str,f64,str
"""cell""",null,"[""GABAergic"", ""Glutamatergic""]",0,null,"""#000000""","""major_class""",null,"""visp_met_types"""
"""GABAergic""","""cell""","[""Sst-MET-1"", ""Sst-MET-2"", … ""Vip-MET-5""]",1,null,"""#EF4136""","""class""",null,"""visp_met_types"""
"""Glutamatergic""","""cell""","[""L2/3 IT"", ""L4 IT"", … ""L6b""]",1,null,"""#27AAE1""","""class""",null,"""visp_met_types"""
"""Sst-MET-1""","""GABAergic""",null,2,null,"""#b9bb67""","""cluster""",null,"""visp_met_types"""
"""Sst-MET-2""","""GABAergic""",null,2,null,"""#804811""","""cluster""",null,"""visp_met_types"""
…,…,…,…,…,…,…,…,…
"""L5 ET-3""","""Glutamatergic""",null,2,null,"""#29E043""","""cluster""",null,"""visp_met_types"""
"""L5 NP""","""Glutamatergic""",null,2,null,"""#73CA95""","""cluster""",null,"""visp_met_types"""
"""L6 CT-1""","""Glutamatergic""",null,2,null,"""#74CAFF""","""cluster""",null,"""visp_met_types"""


# OG Cluster Membership
- __minnie65__ — Has its __own clustering__ (neuron → glutamatergic/gabaergic → subtypes like L4IT, PTC, etc.) under project_id="minnie65". Each cell gets ClusterMembership entries at each hierarchy level.
- __visp_inh_patchseq__ — Has ClusterMembership entries to the __visp_met_types__ taxonomy (MET-type clusters like Sncg-MET-1, etc.).
- __visp_exc_patchseq__ — Also has ClusterMembership entries to __the visp_met_types__ taxonomy.   

__Problem: The "cluster" column will show the children clusters without the cluster_project_id__ but L6b of met types wont be the same as L6b of minnie types

In [9]:
df = pl.read_delta(os.path.join(OUTPUT_ROOT, "clustermembership"))
print(f"Unique project_ids: {df['project_id'].unique().to_list()}")
df

Unique project_ids: ['visp_inh_patchseq', 'minnie65', 'visp_exc_patchseq']


item,cluster,membership_score,probability,distance,project_id
str,str,f64,f64,f64,str
"""1039273993""","""L6b""",null,null,null,"""visp_exc_patchseq"""
"""1039273993""","""Glutamatergic""",null,null,null,"""visp_exc_patchseq"""
"""1039273993""","""cell""",null,null,null,"""visp_exc_patchseq"""
"""823218199""","""L5 ET-3""",null,null,null,"""visp_exc_patchseq"""
"""823218199""","""Glutamatergic""",null,null,null,"""visp_exc_patchseq"""
…,…,…,…,…,…
"""993245688""","""GABAergic""",null,null,null,"""visp_inh_patchseq"""
"""993245688""","""cell""",null,null,null,"""visp_inh_patchseq"""
"""993283588""","""Sncg-MET-1""",null,null,null,"""visp_inh_patchseq"""


## Cell to cluster mapping (cross-taxonomy mapping)
- __visp_inh_patchseq__ → mapped to __tasic_2018_visp_scrnaseq__ (T-type mapping via "Tree mapping" method), under __mapping set visp_inh_patchseq_ttype_mapping__.
- __visp_exc_patchseq__ → mapped to __tasic_2018_visp_scrnaseq__ (T-type mapping via "Tree mapping" method), under __mapping set visp_exc_patchseq_ttype_mapping__.
- __visp_exc_wnm__ → mapped to __visp_met_types__ under mapping set __visp_exc_wnm_mettype_mapping__, using "Routed random forest mapping". This includes a probability score per mapping.

In [11]:
df = pl.read_delta(os.path.join(OUTPUT_ROOT, "celltoclustermapping"))
print(f"Unique project_ids: {df['project_id'].unique().to_list()}")
print(f"Unique mapping_sets: {df['mapping_set'].unique().to_list()}")
df

Unique project_ids: ['visp_inh_patchseq', 'visp_exc_wnm', 'visp_exc_patchseq']
Unique mapping_sets: ['visp_exc_wnm_mettype_mapping', 'visp_inh_patchseq_ttype_mapping', 'visp_exc_patchseq_ttype_mapping']


id,mapping_set,source_cell,target_cluster,score,probability,notes,project_id
str,str,str,str,f64,f64,str,str
"""182709_6984-X2452-Y12423_reg-L…","""visp_exc_wnm_mettype_mapping""","""182709_6984-X2452-Y12423_reg""","""L5 ET-2""",null,0.988,null,"""visp_exc_wnm"""
"""182709_6984-X2452-Y12423_reg-G…","""visp_exc_wnm_mettype_mapping""","""182709_6984-X2452-Y12423_reg""","""Glutamatergic""",null,null,null,"""visp_exc_wnm"""
"""182709_6984-X2452-Y12423_reg-c…","""visp_exc_wnm_mettype_mapping""","""182709_6984-X2452-Y12423_reg""","""cell""",null,null,null,"""visp_exc_wnm"""
"""182709_7126-X2913-Y10535_reg-L…","""visp_exc_wnm_mettype_mapping""","""182709_7126-X2913-Y10535_reg""","""L5 ET-3""",null,0.918,null,"""visp_exc_wnm"""
"""182709_7126-X2913-Y10535_reg-G…","""visp_exc_wnm_mettype_mapping""","""182709_7126-X2913-Y10535_reg""","""Glutamatergic""",null,null,null,"""visp_exc_wnm"""
…,…,…,…,…,…,…,…
"""645369849-cell-visp_exc_patchs…","""visp_exc_patchseq_ttype_mappin…","""645369849""","""cell""",null,null,null,"""visp_exc_patchseq"""
"""831254527-L6 CT VISp Ctxn3 Sla…","""visp_exc_patchseq_ttype_mappin…","""831254527""","""L6 CT VISp Ctxn3 Sla""",null,null,null,"""visp_exc_patchseq"""
"""831254527-L6 CT-visp_exc_patch…","""visp_exc_patchseq_ttype_mappin…","""831254527""","""L6 CT""",null,null,null,"""visp_exc_patchseq"""


---